kernel : Python (Pixi)

#### Bioproject: PRJNA1287730 (Reference Genome: T2T_CHM13_v2_0)

In [5]:
import gc
import os
import pandas as pd
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

def load_rds_data(rds_path: str, series: str) -> AnnData:
    import anndata2ri
    from rpy2.robjects import r
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    importr("base")
    importr("Matrix")
    importr("Seurat")
    importr("SingleCellExperiment")

    rcode = f"""
    obj <- readRDS("{rds_path}")
    if (inherits(obj, "Seurat")) {{
        obj <- as.SingleCellExperiment(obj)
    }}

    if (!inherits(obj, "SingleCellExperiment")) {{
        stop("Loaded object is not a SingleCellExperiment (or derived from it).")
    }}

    anames <- assayNames(obj)
    use_assay <- if ("counts" %in% anames) "counts" else anames[[1]]

    mat <- assay(obj, use_assay)
    mat <- as(mat, "dgCMatrix")

    assay(obj, "counts") <- mat
    # Overwrite 'X' assay with 'counts' assay
    assay(obj, "X") <- assay(obj, "counts")
    obj <- as(obj, "SingleCellExperiment") # Ensure the object is a SingleCellExperiment
    obj
    """
    with localconverter(anndata2ri.converter):
        adata = r(rcode)
        r("if (exists('obj')) rm(obj)")
        r("gc()")

    if not isinstance(adata, AnnData):
        raise ValueError(f"Result is not AnnData. Got type: {type(adata)}")

    adata.layers.pop("counts", None)
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    
    # Multidimensional representations (obsm, varm, layers) are intentionally cleared at this stage,
    # since integrated multi-modal / multi-dimensional analyses will be performed after dataset integration, 
    # overwriting any existing representations.
    adata.obsm = {}
    adata.varm = {}
    adata.layers = {}
    gc.collect()

    adata.obs["series"] = series
    return adata

In [6]:
FILE = "GSE301908_sn_all.rds"
SERIES = "feng"
SAVE_FILE_NAME = SERIES + "_raw.h5ad"

data_dir = os.path.join(raw_data_dir, SERIES)
adata = load_rds_data(os.path.join(data_dir, FILE), series = SERIES)
adata.obs.head() #type: ignore

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: Layer ‘counts’ is empty 
  
R callback write-console: 2: Layer ‘scale.data’ is empty 
  


,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,majorCluster,patient_info,ident,series
ms10r0_AAACCCAAGAACTCCT-1,ms10r0,4788.0,2050.0,0.382217,Oligo,patient,Oligo,feng
ms10r0_AAACCCAAGAGGATGA-1,ms10r0,8882.0,3091.0,0.338465,Oligo,patient,Oligo,feng
ms10r0_AAACCCAAGCATGCGA-1,ms10r0,3063.0,1185.0,0.380108,Oligo,patient,Oligo,feng
ms10r0_AAACCCACAAGAGTAT-1,ms10r0,13686.0,3884.0,0.657662,Oligo,patient,Oligo,feng
ms10r0_AAACCCACAATAGTAG-1,ms10r0,8140.0,3168.0,0.961653,Oligo,patient,Oligo,feng


In [13]:
adata.obs["patient_info"].value_counts()

patient_info
patient    231435
ctrl        62535
Name: count, dtype: int64

In [14]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["sample"] = adata.obs["orig.ident"]
adata.obs["donor"] = adata.obs["sample"]
adata.obs["celltype"] = adata.obs["majorCluster"]
adata.obs["condition"] = adata.obs["patient_info"]
adata.obs["series"] = adata.obs["series"]

keep = ["sample", "donor", "celltype", "condition", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [ ]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE_NAME))
del adata
gc.collect()

3769